# MAPPO Curriculum

This notebook trains the first curriculum stage for `AntByteForagingEnv`: ants learn to reach cookie sources, pick up bites, return to the hub, and write configurable tile values into the environment. The trainer predicts both movement and write-value actions for every ant; `WRITE_BITS = 1` is the current curriculum setting and can be raised to 3, 5, or up to 8 later. Training now uses the pure JAX MAPPO path: JAX environment rollouts, JIT-compiled GAE/PPO updates, local actor observations, and a centralized critic.

The training cell below grows the map progressively. It pads observations to the largest scheduled map, so the same actor/critic checkpoint can continue from smaller maps to larger maps. Each episode randomizes the colony location and uses multiple random cookie source locations.

Install the notebook/training extras from the repo root if needed:

```bash
python -m pip install -e ".[jax,notebooks]"
```


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import sys

try:
    import jax  # noqa: F401
    import jax.numpy as jnp  # noqa: F401
    import imageio.v2 as imageio  # noqa: F401
    import tqdm  # noqa: F401
except ModuleNotFoundError as exc:
    missing = exc.name or "jax/notebook extras"
    raise ModuleNotFoundError(
        f"Missing {missing!r} in this notebook kernel ({sys.executable}). "
        "Install the JAX notebook extras from the repo root with: "
        f'{sys.executable} -m pip install -e ".[jax,notebooks]"'
    ) from exc


In [ ]:
import importlib
import pickle
from types import SimpleNamespace

import imageio.v2 as imageio
import jax
import jax.numpy as jnp
import numpy as np
from tqdm.auto import tqdm

from ant_byte_env import AntByteForagingEnv
from ant_byte_env.vault import create_vault_entry
import train_mappo_jax

importlib.reload(train_mappo_jax)
from train_mappo_jax import (
    build_actor_observations,
    build_central_observations,
    flatten_agent_actions,
    get_action_and_value,
    main,
)


def _sample_hub_position(args, rng):
    if getattr(args, "random_hub", False):
        return (
            int(rng.integers(0, args.width)),
            int(rng.integers(0, args.height)),
        )
    return (args.width // 2, args.height // 2)


def build_curriculum_reset_options(args, *, seed=None):
    rng = np.random.default_rng(seed)
    hub = _sample_hub_position(args, rng)
    if getattr(args, "random_food", False):
        return {"hub_pos": hub}

    distance = min(args.cookie_distance, max(args.width, args.height))
    offsets = ((distance, 0), (-distance, 0), (0, distance), (0, -distance))
    for x_offset, y_offset in offsets:
        candidate = (hub[0] + x_offset, hub[1] + y_offset)
        if 0 <= candidate[0] < args.width and 0 <= candidate[1] < args.height:
            return {"hub_pos": hub, "food_positions": [candidate]}

    for y_pos in range(args.height):
        for x_pos in range(args.width):
            if (x_pos, y_pos) != hub:
                return {"hub_pos": hub, "food_positions": [(x_pos, y_pos)]}

    return {"hub_pos": hub}


def obs_to_jax_batch(obs):
    return {key: jnp.asarray(value[None, ...]) for key, value in obs.items()}


def draw_vision_squares(
    frame,
    obs,
    *,
    tile_size,
    vision_radius,
    colors=((61, 220, 255), (255, 113, 206), (255, 226, 89), (93, 255, 139)),
    border_px=2,
    fill_alpha=0.12,
):
    output = frame.copy()
    grid_height, grid_width = obs["food"].shape
    frame_height, frame_width = output.shape[:2]

    for ant_index, position in enumerate(obs["ants_pos"]):
        x_pos, y_pos = int(position[0]), int(position[1])
        left_tile = max(0, x_pos - vision_radius)
        right_tile = min(grid_width, x_pos + vision_radius + 1)
        top_tile = max(0, y_pos - vision_radius)
        bottom_tile = min(grid_height, y_pos + vision_radius + 1)
        x0 = int(np.clip(left_tile * tile_size, 0, frame_width))
        x1 = int(np.clip(right_tile * tile_size, 0, frame_width))
        y0 = int(np.clip(top_tile * tile_size, 0, frame_height))
        y1 = int(np.clip(bottom_tile * tile_size, 0, frame_height))
        if x0 >= x1 or y0 >= y1:
            continue

        color = np.array(colors[ant_index % len(colors)], dtype=np.float32)
        region = output[y0:y1, x0:x1].astype(np.float32)
        output[y0:y1, x0:x1] = (
            region * (1.0 - fill_alpha) + color * fill_alpha
        ).astype(output.dtype)

        border = min(border_px, max(1, (x1 - x0) // 2), max(1, (y1 - y0) // 2))
        output[y0 : y0 + border, x0:x1] = color.astype(output.dtype)
        output[y1 - border : y1, x0:x1] = color.astype(output.dtype)
        output[y0:y1, x0 : x0 + border] = color.astype(output.dtype)
        output[y0:y1, x1 - border : x1] = color.astype(output.dtype)

    return output


## Quick Smoke Run

Run this first to make sure the notebook kernel can import the repo, create the JAX environment/trainer path, and complete one tiny JAX MAPPO update.


In [ ]:
smoke_metrics = main(
    [
        "--total-timesteps", "8",
        "--num-envs", "1",
        "--num-steps", "4",
        "--num-minibatches", "1",
        "--update-epochs", "1",
        "--width", "4",
        "--height", "4",
        "--num-ants", "1",
        "--food-count", "1",
        "--max-steps", "8",
        "--write-bits", "1",
        "--hidden-size", "16",
        "--seed", "11",
        "--quiet",
    ]
)
smoke_metrics


## Progressive Map Curriculum

Each stage resumes from the previous stage checkpoint. Keep `--obs-width` and `--obs-height` equal to the largest scheduled map, otherwise the padded critic input size will not match the saved checkpoint.

The default schedule trains every square map size from `4x4` through `15x15`. It increases `cookie_distance`, uses a moderate food count, grows the number of food sources, randomizes the hub, and keeps `WRITE_BITS = 1` for now.

Each stage trains for exactly `GLOBAL_UPDATE_CAP` JAX MAPPO updates. After each stage finishes, the notebook immediately renders that policy, saves the rollout video, archives it in the vault, and then continues to the next map.


In [ ]:
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def curriculum_food_count(size):
    return 2 + max(0, size - 4)


def curriculum_food_sources(size):
    return min(curriculum_food_count(size), max(2, size // 2))


CURRICULUM_STAGES = [
    {
        "name": f"{size}x{size}",
        "width": size,
        "height": size,
        "food_count": curriculum_food_count(size),
        "food_sources": curriculum_food_sources(size),
        "cookie_distance": min(1 + (size - 4) // 2, size // 2),
        "max_steps": max(48, 4 * size * size),
    }
    for size in range(4, 16)
]

MAX_WIDTH = max(stage["width"] for stage in CURRICULUM_STAGES)
MAX_HEIGHT = max(stage["height"] for stage in CURRICULUM_STAGES)
NUM_ENVS = 64
NUM_STEPS = 80
UPDATE_TIMESTEPS = NUM_ENVS * NUM_STEPS
GLOBAL_UPDATE_CAP = 100
ACTOR_VISION_RADIUS = 2
WRITE_BITS = 1
ROLLOUT_TILE_SIZE = 32
print(jax.devices()[0])

COMMON_ARGS = [
    "--num-envs", str(NUM_ENVS),
    "--num-steps", str(NUM_STEPS),
    "--num-minibatches", "4",
    "--update-epochs", "4",
    "--obs-width", str(MAX_WIDTH),
    "--obs-height", str(MAX_HEIGHT),
    "--actor-vision-radius", str(ACTOR_VISION_RADIUS),
    "--write-bits", str(WRITE_BITS),
    "--num-ants", "1",
    "--random-food",
    "--random-hub",
    "--pickup-bonus", "0.25",
    "--distance-bonus", "0.02",
    "--hidden-size", "128",
    "--seed", "1",
    "--quiet",
]


def load_jax_checkpoint(checkpoint_path):
    with checkpoint_path.open("rb") as checkpoint_file:
        checkpoint = pickle.load(checkpoint_file)
    checkpoint["params"] = jax.tree_util.tree_map(jnp.asarray, checkpoint["params"])
    return checkpoint


def render_policy_rollout(checkpoint_path):
    checkpoint = load_jax_checkpoint(checkpoint_path)
    saved_args = SimpleNamespace(**checkpoint["args"])
    write_bits = getattr(saved_args, "write_bits", WRITE_BITS)
    vision_radius = saved_args.actor_vision_radius
    params = checkpoint["params"]
    action_key = jax.random.PRNGKey(saved_args.seed + 100_000)

    env = AntByteForagingEnv(
        width=saved_args.width,
        height=saved_args.height,
        num_ants=saved_args.num_ants,
        food_count=saved_args.food_count,
        food_source_count=saved_args.food_sources,
        max_steps=saved_args.max_steps,
        random_food=saved_args.random_food,
        render_mode="rgb_array",
        tile_size=ROLLOUT_TILE_SIZE,
        write_bits=write_bits,
    )

    frames = []
    try:
        obs, info = env.reset(
            seed=saved_args.seed,
            options=build_curriculum_reset_options(saved_args, seed=saved_args.seed),
        )
        frame = env.render()
        if frame is not None:
            frames.append(
                draw_vision_squares(
                    frame,
                    obs,
                    tile_size=ROLLOUT_TILE_SIZE,
                    vision_radius=vision_radius,
                )
            )

        for _ in tqdm(
            range(saved_args.max_steps),
            desc=f"{checkpoint_path.stem} rollout",
            leave=False,
        ):
            obs_batch = obs_to_jax_batch(obs)
            central_obs = build_central_observations(
                obs_batch,
                food_scale=saved_args.food_count,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )
            actor_obs = build_actor_observations(
                obs_batch,
                food_scale=saved_args.food_count,
                actor_vision_radius=saved_args.actor_vision_radius,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )
            action_key, step_key = jax.random.split(action_key)
            joint_actions, _, _, _ = get_action_and_value(
                params,
                actor_obs,
                central_obs,
                step_key,
                deterministic=False,
            )

            env_action = np.asarray(flatten_agent_actions(joint_actions))[0]
            obs, reward, terminated, truncated, info = env.step(env_action)
            frame = env.render()
            if frame is not None:
                frames.append(
                    draw_vision_squares(
                        frame,
                        obs,
                        tile_size=ROLLOUT_TILE_SIZE,
                        vision_radius=vision_radius,
                    )
                )
            if terminated or truncated:
                break
    finally:
        env.close()

    if not frames:
        raise RuntimeError(f"No frames were rendered for {checkpoint_path}.")

    video_path = checkpoint_path.with_name(f"{checkpoint_path.stem}_rollout.mp4")
    imageio.mimsave(video_path, frames, fps=AntByteForagingEnv.metadata["render_fps"])
    return video_path


stage_metrics = []
stage_video_paths = []
stage_vault_entries = []
previous_checkpoint = None

for stage_index, stage in enumerate(CURRICULUM_STAGES, start=1):
    print(f"Training stage {stage_index}/{len(CURRICULUM_STAGES)}: {stage['name']}")
    checkpoint_path = CHECKPOINT_DIR / f"jax_mappo_forage_stage1_{stage['name']}.pkl"
    load_checkpoint = previous_checkpoint

    update_iterator = tqdm(
        range(1, GLOBAL_UPDATE_CAP + 1),
        total=GLOBAL_UPDATE_CAP,
        desc=f"{stage['name']}",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )
    for update_index in update_iterator:
        train_args = [
            *COMMON_ARGS,
            "--total-timesteps", str(UPDATE_TIMESTEPS),
            "--width", str(stage["width"]),
            "--height", str(stage["height"]),
            "--food-count", str(stage["food_count"]),
            "--food-sources", str(stage["food_sources"]),
            "--cookie-distance", str(stage["cookie_distance"]),
            "--max-steps", str(stage["max_steps"]),
            "--save-model", str(checkpoint_path),
        ]
        if load_checkpoint is not None:
            train_args.extend(["--load-model", str(load_checkpoint)])

        train_metrics = main(train_args)
        update_iterator.set_postfix(
            loss=f"{train_metrics['loss']:.3f}",
            ret=f"{train_metrics['episode_return']:.3f}",
        )
        metrics_row = {
            **stage,
            **train_metrics,
            "stage_update": update_index,
            "global_update_cap": GLOBAL_UPDATE_CAP,
            "checkpoint": str(checkpoint_path),
        }
        stage_metrics.append(metrics_row)

        load_checkpoint = checkpoint_path

    video_path = render_policy_rollout(checkpoint_path)
    vault_entry_path = create_vault_entry(
        vault_dir=PROJECT_ROOT / "vault",
        title=f"JAX MAPPO policy rollout {stage['name']}",
        description=f"Rollout video after training stage {stage['name']} for {GLOBAL_UPDATE_CAP} JAX MAPPO updates.",
        assets=[video_path],
        metadata={
            "stage": stage,
            "checkpoint_path": str(checkpoint_path),
            "video_path": str(video_path),
            "actor_vision_radius": ACTOR_VISION_RADIUS,
            "write_bits": WRITE_BITS,
            "global_update_cap": GLOBAL_UPDATE_CAP,
        },
    )
    stage_video_paths.append(video_path)
    stage_vault_entries.append(vault_entry_path)

    print(f"Saved rollout video to {video_path}")
    print(f"Archived rollout in {vault_entry_path}")

    previous_checkpoint = checkpoint_path

FINAL_CHECKPOINT_PATH = previous_checkpoint
{
    "stage_metrics": stage_metrics,
    "stage_video_paths": stage_video_paths,
    "stage_vault_entries": stage_vault_entries,
}


## Optional Re-Render Rollouts

The training cell already renders and archives each stage immediately after its configured updates. Run this optional cell only if you want to regenerate videos from existing JAX checkpoints without retraining.


In [ ]:
policy_checkpoint_paths = [
    CHECKPOINT_DIR / f"jax_mappo_forage_stage1_{stage['name']}.pkl"
    for stage in CURRICULUM_STAGES
]
missing_checkpoints = [path for path in policy_checkpoint_paths if not path.exists()]
if missing_checkpoints:
    missing = "\n".join(str(path) for path in missing_checkpoints)
    raise FileNotFoundError(f"Train the missing stage policies before rendering:\n{missing}")

video_paths = [
    render_policy_rollout(path)
    for path in tqdm(policy_checkpoint_paths, desc="rendering policies")
]
vault_entry_path = create_vault_entry(
    vault_dir=PROJECT_ROOT / "vault",
    title="JAX MAPPO curriculum policy rollouts",
    description="Rollout videos for each saved JAX MAPPO curriculum stage policy.",
    assets=video_paths,
    metadata={
        "stages": [stage["name"] for stage in CURRICULUM_STAGES],
        "checkpoint_paths": [str(path) for path in policy_checkpoint_paths],
        "actor_vision_radius": ACTOR_VISION_RADIUS,
        "write_bits": WRITE_BITS,
        "global_update_cap": GLOBAL_UPDATE_CAP,
    },
)
{
    "video_paths": video_paths,
    "vault_entry_path": vault_entry_path,
}
